In [1]:
!pip install -q langchain-openai langchain-community langchain-core python-dotenv
print("Librerías instaladas.")

Librerías instaladas.


In [2]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

llm = ChatOpenAI(
    base_url="https://models.inference.ai.azure.com",
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o",
    temperature=0.3
)

if os.getenv("GITHUB_TOKEN"):
    print(f"Conectado exitosamente. Token: {os.getenv('GITHUB_TOKEN')[:4]}...")
else:
    print("Error: No se encontró GITHUB_TOKEN")

Conectado exitosamente. Token: gith...


In [3]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 1. Almacén de memoria
store = {}
def get_session_history(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# 2. Prompt
prompt = ChatPromptTemplate.from_messages([
    ("system", """Eres el Asistente de Soporte de DuocUC. 
    
    ### TÉCNICA: CHAIN-OF-THOUGHT (CoT)
    Para cada mensaje, sigue estos pasos mentalmente antes de responder:
    1. Identifica la emoción del alumno.
    2. Determina si el problema es de software, hardware o administrativo.
    3. Verifica en el historial si el alumno ya dio su nombre o datos.

    ### TÉCNICA: FEW-SHOT (Ejemplos)
    Alumno: "No me carga el portal." -> Respuesta: Entiendo tu frustración. ¿Has probado borrar las cookies? Categoría: TI.
    Alumno: "¿Dónde pido mi certificado?" -> Respuesta: Hola. Debes ir a la oficina de Registro Curricular. Categoría: Administrativo.

    ### RESTRICCIÓN:
    Si el alumno ya te dijo su nombre en el historial (memoria), úsalo para saludarlo."""),
    
    #aqui se inserta la memoria automaticamente 
    MessagesPlaceholder(variable_name="history"),
    
    ("human", "{input}"),
])

# 3. La Cadena (Chain)
chain = prompt | llm

# 4. Envoltura con historial
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history",
)

print("Todo integrado: Rol, CoT, Few-Shot y Memoria en una sola cadena.")

Todo integrado: Rol, CoT, Few-Shot y Memoria en una sola cadena.


In [4]:
# Celda 4: Interacción y Pruebas de Memoria
def chatear_con_asistente(texto_usuario, id_sesion="chat_duoc_01"):
    """
    Función principal para interactuar con la IA manteniendo el contexto.
    """
    # Configuramos la sesión para que la memoria sepa a qué chat pertenece
    config = {"configurable": {"session_id": id_sesion}}
    
    # Invocamos la cadena que creamos en la Celda 3
    # chain_with_history se encarga de:
    # 1. Buscar el historial (Memoria)
    # 2. Aplicar el Role + CoT + Few-Shot (Prompt Engineering)
    # 3. Llamar al modelo (LLM)
    respuesta = chain_with_history.invoke(
        {"input": texto_usuario}, 
        config=config
    )
    
    return respuesta.content

# --- PRUEBA DE FUNCIONAMIENTO ---
print("Iniciando chat de prueba...\n")

# Paso 1: Presentación (Zero-shot + Memoria)
msg1 = chatear_con_asistente("Hola, soy Ignacio Salazar de la carrera de Informática.")
print(f"Alumno: Hola, soy Ignacio...\nAsistente: {msg1}\n")

# Paso 2: Prueba de Contexto (Verificar si recuerda el nombre)
msg2 = chatear_con_asistente("¿Podrías decirme qué carrera estudio y qué ejemplos de ayuda me diste antes?")
print(f"Alumno: ¿Recuerdas mi carrera?\nAsistente: {msg2}")

Iniciando chat de prueba...

Alumno: Hola, soy Ignacio...
Asistente: ¡Hola, Ignacio! ¿En qué puedo ayudarte hoy? 😊

Alumno: ¿Recuerdas mi carrera?
Asistente: Claro, Ignacio. Estudias la carrera de Informática. En cuanto a los ejemplos de ayuda que mencioné antes, te di uno relacionado con problemas técnicos ("No me carga el portal") y otro administrativo ("¿Dónde pido mi certificado?"). ¿Hay algo más en lo que pueda asistirte? 😊
